# Navigator context MCP — demo

This notebook wires the **navigator_context** MCP server into the OpenAI Agents SDK. The server keeps a **problem statement**, a **success metric**, and **session notes**, then can emit a **briefing** string you can fold into prompts—useful when you want explicit, inspectable context instead of hiding everything inside model memory.

**Requirements:** `OPENAI_API_KEY` in your environment (e.g. `.env` at the repo root). Run `uv sync` at the repo root if needed.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display

load_dotenv(override=True)


def find_server_dir() -> Path:
    here = Path.cwd().resolve()
    if (here / "navigator_server.py").exists():
        return here
    rel = here / "6_mcp/community_contributions/erisanolasheni/navigator_context_mcp"
    if rel.is_dir() and (rel / "navigator_server.py").exists():
        return rel
    raise FileNotFoundError(
        "Could not find navigator_server.py. Run Jupyter from the repo root or from this folder."
    )


SERVER_DIR = find_server_dir()
print("MCP cwd:", SERVER_DIR)

In [ ]:
params = {
    "command": "uv",
    "args": ["run", "navigator_server.py"],
    "cwd": str(SERVER_DIR),
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    tools = await server.list_tools()
    print([t.name for t in tools])

## Agent that uses the navigator tools

The agent can set the problem and metric, log what it tried, and fetch the briefing. Keep instructions small and trace the run in the SDK UI.

In [ ]:
instructions = """You help a builder clarify an agentic project using the navigator tools.

1. If the user describes a vague goal, ask one short clarifying question OR infer a crisp problem and record it with set_problem_statement.
2. Propose one concrete success metric and save it with set_success_metric.
3. Log what you did with append_session_note.
4. End by calling get_briefing_for_prompt and quoting that briefing in your final reply so the user sees the stored context.
"""

model = "gpt-4o-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp:
    agent = Agent(
        name="Navigator assistant",
        instructions=instructions,
        model=model,
        mcp_servers=[mcp],
    )
    with trace("Navigator MCP demo"):
        result = await Runner.run(
            agent,
            "I want something that helps our team run better standups without another heavy app.",
        )

display(Markdown(result.final_output))